# Graficos de metricas de execucao

Este notebook le os relatorios gerados por:

```bash
make profile-chunk-data
```

ou:

```bash
make profile-command COMMAND="poetry run python -B src/rag/index_hoi4_qdrant.py"
```

Ele procura execucoes em `metrics/*/samples.csv` e gera graficos em `avaliacao/outputs/metricas/`.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)

sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="tab10",
    font_scale=1.05,
)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#fbfbf7"

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Makefile").exists() and (candidate / "scripts" / "monitor_command.py").exists():
            return candidate
    raise FileNotFoundError("Nao encontrei a raiz do projeto a partir do diretorio atual.")

ROOT = find_project_root(Path.cwd())
METRICS_DIR = ROOT / "metrics"
OUTPUT_DIR = ROOT / "avaliacao" / "outputs" / "metricas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT, METRICS_DIR, OUTPUT_DIR

## 1. Descobrir execucoes disponiveis

In [ ]:
def load_summary(run_dir: Path) -> dict:
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        return {}
    with summary_path.open("r", encoding="utf-8") as file:
        return json.load(file)

def discover_runs(metrics_dir: Path) -> pd.DataFrame:
    rows = []
    if not metrics_dir.exists():
        return pd.DataFrame(rows)

    for samples_path in sorted(metrics_dir.glob("*/samples.csv")):
        run_dir = samples_path.parent
        summary = load_summary(run_dir)
        rows.append({
            "run_id": run_dir.name,
            "run_dir": run_dir,
            "samples_path": samples_path,
            "summary_path": run_dir / "summary.json",
            "stdout_path": run_dir / "stdout.log",
            "stderr_path": run_dir / "stderr.log",
            "started_at": summary.get("started_at"),
            "finished_at": summary.get("finished_at"),
            "elapsed_seconds": summary.get("elapsed_seconds"),
            "returncode": summary.get("returncode"),
            "peak_process_cpu_percent": summary.get("peak_process_cpu_percent"),
            "peak_system_cpu_percent": summary.get("peak_system_cpu_percent"),
            "peak_rss_mb": summary.get("peak_rss_mb"),
            "max_disk_read_mb": summary.get("max_disk_read_mb"),
            "max_disk_write_mb": summary.get("max_disk_write_mb"),
            "gpu_available": summary.get("gpu_available"),
            "peak_gpu_util_percent": summary.get("peak_gpu_util_percent"),
            "peak_gpu_memory_used_mb": summary.get("peak_gpu_memory_used_mb"),
            "command": " ".join(summary.get("command", [])),
        })

    runs = pd.DataFrame(rows)
    if not runs.empty:
        runs["started_at"] = pd.to_datetime(runs["started_at"], errors="coerce")
        runs["finished_at"] = pd.to_datetime(runs["finished_at"], errors="coerce")
        runs = runs.sort_values("started_at", na_position="last").reset_index(drop=True)
    return runs

runs = discover_runs(METRICS_DIR)
if runs.empty:
    print("Nenhuma execucao encontrada em metrics/. Gere dados com: make profile-chunk-data")
else:
    display(runs[["run_id", "started_at", "elapsed_seconds", "returncode", "peak_rss_mb", "peak_process_cpu_percent", "command"]])

## 2. Escolher execucao

Por padrao, o notebook usa a execucao mais recente. Para analisar outra, altere `RUN_INDEX`.

In [ ]:
RUN_INDEX = -1

if runs.empty:
    samples = pd.DataFrame()
    selected_run = None
else:
    selected_run = runs.iloc[RUN_INDEX]
    samples = pd.read_csv(selected_run["samples_path"])
    samples["timestamp"] = pd.to_datetime(samples["timestamp"], errors="coerce")
    samples = samples.sort_values("elapsed_seconds").reset_index(drop=True)
    print(f"Execucao selecionada: {selected_run['run_id']}")
    display(samples.head())

## 3. Funcoes de grafico

In [ ]:
SMOOTH_WINDOW = 5


def safe_filename(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9._-]+", "-", value)
    return value.strip("-") or "grafico"


def save_current_figure(name: str) -> Path:
    path = OUTPUT_DIR / f"{safe_filename(name)}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path


def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(df[column], errors="coerce")


def run_palette(df: pd.DataFrame) -> dict:
    run_ids = sorted(df["run_id"].dropna().astype(str).unique())
    colors = sns.color_palette("tab20", n_colors=max(len(run_ids), 1))
    return dict(zip(run_ids, colors))


def add_smoothed_value(df: pd.DataFrame, value_column: str = "value", window: int = SMOOTH_WINDOW) -> pd.DataFrame:
    if df.empty:
        return df
    smoothed = df.sort_values(["run_id", "elapsed_seconds"]).copy()
    smoothed["value_smoothed"] = (
        smoothed.groupby("run_id", group_keys=False)[value_column]
        .transform(lambda series: series.rolling(window=window, min_periods=1, center=True).mean())
    )
    return smoothed


def load_all_samples(runs_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, run in runs_df.iterrows():
        current = pd.read_csv(run["samples_path"])
        current.insert(0, "run_id", run["run_id"])
        current.insert(1, "run_started_at", run["started_at"])
        current["timestamp"] = pd.to_datetime(current["timestamp"], errors="coerce")
        rows.append(current)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def plot_metric_comparison(df: pd.DataFrame, y: str, title: str, ylabel: str, filename: str) -> None:
    if df.empty or y not in df.columns:
        print(f"Sem dados para {y}.")
        return

    plot_df = df[["run_id", "elapsed_seconds", y]].copy()
    plot_df[y] = numeric_series(plot_df, y)
    plot_df = plot_df.dropna(subset=[y])
    if plot_df.empty:
        print(f"Coluna {y} sem valores numericos.")
        return

    plot_df = plot_df.rename(columns={y: "value"})
    plot_df = add_smoothed_value(plot_df)

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.lineplot(
        data=plot_df,
        x="elapsed_seconds",
        y="value_smoothed",
        hue="run_id",
        palette=run_palette(plot_df),
        linewidth=2.4,
        ax=ax,
    )
    ax.set_title(f"{title} - curva suavizada", fontsize=14, weight="bold", loc="left")
    ax.set_xlabel("Tempo desde o inicio da execucao (s)")
    ax.set_ylabel(ylabel)
    ax.legend(title="Execucao", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    sns.despine(ax=ax)
    path = save_current_figure(filename)
    plt.show()
    print(f"Salvo em: {path}")


## 4. Linha do tempo da execucao selecionada

In [ ]:
all_samples_df = load_all_samples(runs)

if all_samples_df.empty:
    print("Sem execucoes para comparar.")
else:
    plot_metric_comparison(all_samples_df, "process_cpu_percent", "CPU do processo por execucao", "CPU do processo (%)", "comparacao-cpu-processo")
    plot_metric_comparison(all_samples_df, "system_cpu_percent", "CPU do sistema por execucao", "CPU do sistema (%)", "comparacao-cpu-sistema")
    plot_metric_comparison(all_samples_df, "rss_mb", "Memoria RSS por execucao", "Memoria RSS (MB)", "comparacao-memoria-rss")
    plot_metric_comparison(all_samples_df, "disk_read_mb", "Leitura de disco por execucao", "Leitura acumulada (MB)", "comparacao-disco-leitura")
    plot_metric_comparison(all_samples_df, "disk_write_mb", "Escrita de disco por execucao", "Escrita acumulada (MB)", "comparacao-disco-escrita")


In [ ]:
if all_samples_df.empty:
    print("Sem execucoes para comparar.")
elif "gpu_available" in all_samples_df.columns and all_samples_df["gpu_available"].astype(str).str.lower().eq("true").any():
    plot_metric_comparison(all_samples_df, "gpu_util_percent_max", "GPU util por execucao", "GPU util max (%)", "comparacao-gpu-util")
    plot_metric_comparison(all_samples_df, "gpu_memory_used_mb_sum", "GPU memoria por execucao", "GPU memoria usada total (MB)", "comparacao-gpu-memoria")
else:
    print("Sem GPU detectada nas amostras das execucoes.")


## 5. Painel combinado

Este painel facilita enxergar correlacoes entre CPU, memoria, disco e GPU no mesmo intervalo de tempo.

In [ ]:
if all_samples_df.empty:
    print("Sem execucoes para comparar.")
else:
    metrics = [
        ("process_cpu_percent", "CPU processo (%)"),
        ("system_cpu_percent", "CPU sistema (%)"),
        ("rss_mb", "RSS (MB)"),
        ("disk_read_mb", "Disco leitura (MB)"),
        ("disk_write_mb", "Disco escrita (MB)"),
        ("gpu_util_percent_max", "GPU util (%)"),
        ("gpu_memory_used_mb_sum", "GPU memoria (MB)"),
    ]

    panel_rows = []
    for column, label in metrics:
        if column not in all_samples_df.columns:
            continue
        current = all_samples_df[["run_id", "elapsed_seconds", column]].copy()
        current[column] = pd.to_numeric(current[column], errors="coerce")
        current = current.dropna(subset=[column])
        if current.empty:
            continue
        current = current.rename(columns={column: "value"})
        current["metric"] = label
        panel_rows.append(current)

    panel_df = pd.concat(panel_rows, ignore_index=True) if panel_rows else pd.DataFrame()
    if panel_df.empty:
        print("Sem metricas numericas para plotar.")
    else:
        panel_df = add_smoothed_value(panel_df)
        grid = sns.relplot(
            data=panel_df,
            x="elapsed_seconds",
            y="value_smoothed",
            hue="run_id",
            row="metric",
            kind="line",
            height=2.3,
            aspect=5.6,
            facet_kws={"sharey": False, "sharex": True},
            linewidth=2.3,
            alpha=0.95,
            palette=run_palette(panel_df),
        )
        grid.set_axis_labels("Tempo desde o inicio da execucao (s)", "")
        grid.set_titles(row_template="{row_name}")
        if grid.legend is not None:
            grid.legend.set_title("Execucao")
        grid.figure.suptitle("Painel comparativo de metricas por execucao - curvas suavizadas", fontsize=15, weight="bold", x=0.02, ha="left")
        grid.figure.subplots_adjust(top=0.94, hspace=0.35)
        path = OUTPUT_DIR / "comparacao-painel-metricas-execucoes.png"
        grid.figure.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Salvo em: {path}")


## 6. Eventos de stdout/stderr

Use esta tabela para cruzar eventos do comando com os graficos de metricas.

In [ ]:
def read_log_events(path: Path, stream: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    pattern = re.compile(r"^(\S+)\s+\[(.*?)\]\s?(.*)$")
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.rstrip("\n")
            match = pattern.match(line)
            if match:
                rows.append({
                    "stream": stream,
                    "line_number": line_number,
                    "timestamp": match.group(1),
                    "label": match.group(2),
                    "message": match.group(3),
                })
            else:
                rows.append({"stream": stream, "line_number": line_number, "timestamp": None, "label": None, "message": line})
    events = pd.DataFrame(rows)
    if not events.empty:
        events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
    return events

if selected_run is not None:
    stdout_events = read_log_events(selected_run["stdout_path"], "stdout")
    stderr_events = read_log_events(selected_run["stderr_path"], "stderr")
    events = pd.concat([stdout_events, stderr_events], ignore_index=True)
    events = events.sort_values(["timestamp", "stream", "line_number"], na_position="last").reset_index(drop=True)
    events_path = OUTPUT_DIR / f"{safe_filename(selected_run['run_id'])}-eventos.csv"
    events.to_csv(events_path, index=False)
    display(events.head(100))
    print(f"Eventos salvos em: {events_path}")

## 7. Comparar execucoes na mesma linha do tempo

Esta secao coloca varias execucoes nos mesmos graficos usando `run_id` como cor. Assim fica mais facil comparar tempo, CPU, memoria, disco e GPU entre configuracoes.


In [ ]:
if runs.empty:
    print("Sem execucoes para comparar.")
else:
    comparison_columns = [
        "run_id",
        "started_at",
        "elapsed_seconds",
        "peak_rss_mb",
        "peak_process_cpu_percent",
        "peak_system_cpu_percent",
        "max_disk_read_mb",
        "max_disk_write_mb",
        "peak_gpu_util_percent",
        "peak_gpu_memory_used_mb",
        "returncode",
    ]
    comparison = runs[comparison_columns].copy()
    display(comparison)


In [ ]:
if "all_samples_df" not in globals():
    all_samples_df = load_all_samples(runs)

if all_samples_df.empty:
    print("Sem amostras para comparar.")
else:
    timeline_metrics = [
        ("process_cpu_percent", "CPU do processo (%)"),
        ("system_cpu_percent", "CPU do sistema (%)"),
        ("rss_mb", "Memoria RSS (MB)"),
        ("disk_read_mb", "Leitura de disco acumulada (MB)"),
        ("disk_write_mb", "Escrita de disco acumulada (MB)"),
        ("gpu_util_percent_max", "GPU util max (%)"),
        ("gpu_memory_used_mb_sum", "GPU memoria usada total (MB)"),
    ]

    plot_rows = []
    for column, label in timeline_metrics:
        if column not in all_samples_df.columns:
            continue
        current = all_samples_df[["run_id", "elapsed_seconds", column]].copy()
        current[column] = pd.to_numeric(current[column], errors="coerce")
        current = current.dropna(subset=[column])
        if current.empty:
            continue
        current = current.rename(columns={column: "value"})
        current["metric"] = label
        plot_rows.append(current)

    timeline_df = pd.concat(plot_rows, ignore_index=True) if plot_rows else pd.DataFrame()
    if timeline_df.empty:
        print("Sem metricas numericas para plotar.")
    else:
        timeline_df = add_smoothed_value(timeline_df)
        grid = sns.relplot(
            data=timeline_df,
            x="elapsed_seconds",
            y="value_smoothed",
            hue="run_id",
            row="metric",
            kind="line",
            height=2.4,
            aspect=5.5,
            facet_kws={"sharey": False, "sharex": True},
            linewidth=2.3,
            alpha=0.95,
            palette=run_palette(timeline_df),
        )
        grid.set_axis_labels("Tempo desde o inicio da execucao (s)", "")
        grid.set_titles(row_template="{row_name}")
        if grid.legend is not None:
            grid.legend.set_title("Execucao")
        grid.figure.suptitle("Linha do tempo comparativa entre execucoes - curvas suavizadas", fontsize=15, weight="bold", x=0.02, ha="left")
        grid.figure.subplots_adjust(top=0.94, hspace=0.35)
        path = OUTPUT_DIR / "comparacao-linha-do-tempo-execucoes.png"
        grid.figure.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Salvo em: {path}")


## 8. Exportar serie consolidada

Este CSV combina as amostras de todas as execucoes e facilita analises externas. Ele e o mesmo dataframe usado nos graficos comparativos acima.


In [ ]:
if "all_samples_df" not in globals() or all_samples_df.empty:
    all_samples_df = load_all_samples(runs)

if not all_samples_df.empty:
    consolidated_path = OUTPUT_DIR / "samples_consolidados.csv"
    all_samples_df.to_csv(consolidated_path, index=False)
    display(all_samples_df.head())
    print(f"Serie consolidada salva em: {consolidated_path}")
else:
    print("Sem amostras para exportar.")
